
# Inferencia en producción y enriquecimiento de etiquetas reales (Churn)

**Autor**: Juan Carlos Alfaro Jiménez

Esta libreta implementa dos responsabilidades complementarias que cierran el ciclo de monitorización del modelo de churn en producción.

La primera es la **inferencia en lote**: carga el *pipeline* `Spark MLlib` registrado bajo el alias `champion` en `Unity Catalog`, lee los clientes que han llegado desde la última ejecución y que aún no tienen predicción en `gold_churn_inference_enriched`, los enriquece con las características del cliente mediante el `PiT` *join* del *feature store* y los transforma con `.transform()` de forma distribuida sobre el clúster de `Databricks`.

La segunda es el **enriquecimiento de etiquetas**: los churns se confirman con retraso una vez que el cliente cancela efectivamente su cuenta o contrato. Cuando una etiqueta real llega a `silver_churn_events`, se propaga a `gold_churn_inference_enriched` mediante un `MERGE` incremental idempotente. Esta tabla, que combina las características de cada cliente, la predicción del modelo y la etiqueta real confirmada, es el dato de entrada de `Databricks Lakehouse Monitoring` para calcular métricas de rendimiento y equidad en producción.


## 1. Importaciones y configuración

In [0]:
%pip install databricks-feature-engineering>=0.13.0
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
exec(open("07_Utils.py").read(), globals())

Total rows: 59,865,300
Total columns: 61

Semantic version: 0

Train period: 2019-12-31 → 2022-12-31
Validation period: 2023-01-01 → 2023-12-31
Test period: 2024-01-01 → 2024-12-31

Train rows: 35,832,870
Validation rows: 11,967,580
Test rows: 11,967,800

Numeric (33): ['amount', 'age', 'city_tier', 'num_cards_issued', 'loyalty_points_balance', 'count_tx_1h', 'sum_amount_1h', 'avg_amount_1h', 'distinct_merchants_1h', 'count_cross_border_1h', 'count_tx_24h', 'sum_amount_24h', 'avg_amount_24h', 'max_amount_24h', 'distinct_merchants_24h', 'distinct_countries_24h', 'count_tor_vpn_24h', 'count_3ds_failed_24h', 'count_tx_7d', 'sum_amount_7d', 'avg_amount_7d', 'distinct_merchants_7d', 'distinct_countries_7d', 'distinct_devices_7d', 'count_tx_30d', 'sum_amount_30d', 'avg_amount_30d', 'max_amount_30d', 'min_amount_30d', 'distinct_merchants_30d', 'distinct_countries_30d', 'num_churn_confirmed_30d', 'spend_24h_vs_avg_30d_ratio']
Boolean (7): ['cross_border', 'is_tor_or_vpn', 'ip_country_match', '

In [0]:
exec(open("08_Utils.py").read(), globals())

08_Utils.py script loaded successfully.


In [0]:
exec(open("09_Utils.py").read(), globals())

Profile features (16): ['age', 'age_group', 'gender', 'occupation', 'city_tier', 'income_bracket', 'income_group', 'customer_segment', 'card_type', 'num_cards_issued', 'two_fa_enabled', 'email_verified', 'phone_verified', 'preferred_channel', 'loyalty_points_balance', 'country']
Aggregation features (28): ['count_tx_1h', 'sum_amount_1h', 'avg_amount_1h', 'distinct_merchants_1h', 'count_cross_border_1h', 'count_tx_24h', 'sum_amount_24h', 'avg_amount_24h', 'max_amount_24h', 'distinct_merchants_24h', 'distinct_countries_24h', 'count_tor_vpn_24h', 'count_3ds_failed_24h', 'count_tx_7d', 'sum_amount_7d', 'avg_amount_7d', 'distinct_merchants_7d', 'distinct_countries_7d', 'distinct_devices_7d', 'count_tx_30d', 'sum_amount_30d', 'avg_amount_30d', 'max_amount_30d', 'min_amount_30d', 'distinct_merchants_30d', 'distinct_countries_30d', 'num_churn_confirmed_30d', 'spend_24h_vs_avg_30d_ratio']
Total feature columns: 44

09_Utils.py script loaded successfully.


In [0]:
from datetime import timedelta

from databricks.feature_engineering import FeatureEngineeringClient

from delta.tables import DeltaTable

import mlflow
import mlflow.spark
from mlflow import MlflowClient

from pyspark.ml.functions import vector_to_array
from pyspark.sql import functions as F

In [0]:
notebook_path_raw = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
notebook = Path(notebook_path_raw).name

print(f"Project: {project}, team: {team}, environment: {environment}")
print(f"Notebook: {notebook}")
print(f"Champion model: {uc_model_name}")
print(f"Spine table: {spine_table}")
print(f"Inference enriched table: {inference_enriched_table}")
print(f"Churn labels table: {churn_labels_table}")

Project: credit_card_churn_detection, team: ml_engineering, environment: development
Notebook: 09_Inference_And_Label_Enrichment
Champion model: workspace.credit_card_churn.churn_lr_pipeline
Spine table: workspace.credit_card_churn.gold_churn_spine
Inference enriched table: workspace.credit_card_churn.gold_churn_inference_enriched
Churn labels table: workspace.credit_card_churn.silver_churn_events



## 2. Carga del modelo `champion`

Se carga el *pipeline* `Spark MLlib` registrado bajo el alias `champion` en `Unity Catalog`. La versión concreta se persiste en `gold_churn_inference_enriched` para garantizar la trazabilidad completa entre cada predicción y el modelo que la generó.

In [0]:
mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri("databricks-uc")

client = MlflowClient()
champion_version = client.get_model_version_by_alias(name = uc_model_name, alias = "champion")
champion_model_version = champion_version.version

pipeline_model = mlflow.spark.load_model(f"models:/{uc_model_name}@champion")

print(f"Champion model loaded: {uc_model_name}")
print(f"Version: {champion_model_version}")

Champion model loaded: workspace.credit_card_churn.churn_lr_pipeline
Version: 2



## 3. Inicialización de la tabla de inferencias enriquecidas

La tabla `gold_churn_inference_enriched` debe existir antes de poder leer las clientes ya puntuadas en la sección siguiente. Si es la primera ejecución, se crea vacía con el esquema correcto derivado de un único ejemplo de predicción. En ejecuciones posteriores esta sección no tiene ningún efecto.

Una vez creada, hay que configurar el monitor de `Databricks Lakehouse Monitoring` desde el `Catalog Explorer` siguiendo los pasos descritos en la sección de conclusiones antes de programar la ejecución automática de esta libreta.

In [0]:
table_exists = spark.catalog.tableExists(inference_enriched_table)

if not table_exists:
    (
        spark.table(baseline_table_name)
             .limit(0)
             .write
             .format("delta")
             .mode("overwrite")
             .option("overwriteSchema", "true")
             .saveAsTable(inference_enriched_table)
    )

    spark.sql(
        f"ALTER TABLE {inference_enriched_table} "
        f"SET TBLPROPERTIES ("
        f"'delta.enableChangeDataFeed' = 'true', "
        f"'project' = '{project}', "
        f"'team' = '{team}')"
    )

    spark.sql(f"""
        ALTER TABLE {inference_enriched_table}
        SET TAGS (
            'project' = '{project}',
            'team' = '{team}'
        )
    """)

    spark.sql(
        f"COMMENT ON TABLE {inference_enriched_table} IS "
        f"'Production inference table. Contains raw transaction features, "
        f"model predictions and confirmed churn labels. "
        f"Monitored by `Databricks Lakehouse Monitoring` against `gold_churn_test_baseline`.'"
    )

    print(f"Table created: {inference_enriched_table}")
else:
    print(f"Table already exists: {inference_enriched_table}")

Table created: workspace.credit_card_churn.gold_churn_inference_enriched



## 4. Lectura de clientes pendientes de predicción

Las clientes de producción son aquellas posteriores al `champion_test_end_date`, la fecha máxima de datos que vio el `champion` actual durante alguna de sus fases (entrenamiento, validación o prueba). Este valor se lee de la etiqueta `test_end_date` de la versión del `champion` en `Unity Catalog`, escrita en `08_Production.ipynb` durante la promoción. Se usa esta fecha en lugar de recalcularla desde `gold_churn_training_dataset` para blindar el *pipeline* contra posibles descoordinaciones: si la tabla de entrenamiento se actualiza antes de que termine un ciclo completo de reentrenamiento, el valor recalculado quedaría adelantado respecto al que realmente vio el `champion`, dejando clientes sin puntuar. Se filtra `gold_churn_spine` por esa fecha y se excluyen mediante un `LEFT ANTI JOIN` las clientes que ya tienen predicción en `gold_churn_inference_enriched`. La operación es idempotente: si la libreta falla y se relanza no se duplican predicciones.

In [0]:
champion_test_end_date = champion_version.tags.get("test_end_date")

production_start_date = (
    datetime.strptime(champion_test_end_date, "%Y-%m-%d") + timedelta(days = 1)
).strftime("%Y-%m-%d")

already_scored_df = (
    spark.table(inference_enriched_table)
         .select(customer_id_column)
)

new_spine_df = (
    spark.table(spine_table)
         .filter(F.col(date_column) >= production_start_date)
         .join(already_scored_df, on = customer_id_column, how = "left_anti")
)

n_new = new_spine_df.count()
print(f"Production cutoff date: {production_start_date}")
print(f"New transactions pending prediction: {n_new:,}")

Production cutoff date: 2025-01-01
New transactions pending prediction: 0



## 5. Enriquecimiento con el *feature store*

Las características del cliente no viajan con la cliente: en producción se recuperan en tiempo real desde el *online feature store*. En este entorno se simulan mediante un `PiT` *join* contra `gold_customer_profile` y `gold_customer_aggregations`, replicando exactamente el enriquecimiento que realizó `05_Training_Dataset_Generation` durante el entrenamiento. Esto garantiza que no hay *training-serving skew*: el *pipeline* recibe exactamente el mismo conjunto de características con el que fue ajustado.

In [0]:
fe = FeatureEngineeringClient()

inference_set = fe.create_training_set(
    df = new_spine_df,
    feature_lookups = feature_lookups,
    label = None,
    exclude_columns = exclude_columns
)

new_transactions_df = inference_set.load_df()

print(f"Enriched transactions: {new_transactions_df.count():,}")

Enriched transactions: 0



## 6. Predicción e inserción en la tabla enriquecida

Se aplica `.transform()` del *pipeline* `Spark MLlib` sobre el `DataFrame` de clientes nuevas. Se extrae la probabilidad de churn como columna escalar desde el vector de probabilidades, y se añaden el instante de inferencia y la versión del modelo. La columna `label_will_churn` se fuerza a nulo porque la etiqueta real no es conocida en el momento de la predicción.

El `MERGE` inserta únicamente las filas cuyo `customer_id` no existe aún en la tabla enriquecida.

In [0]:
if n_new > 0:
    # Base columns equal to exactly what came from the feature-store enriched spine,
    # minus the label (which we add fresh as null).
    # This guarantees the output schema matches baseline table.
    base_columns = [column for column in new_transactions_df.columns if column != label_column]

    scored_df = (
        pipeline_model
        .transform(new_transactions_df)
        .withColumn(
            prob_churn_column,
            vector_to_array(F.col(probability_column)).getItem(1)
        )
        .withColumn(inference_timestamp_col, F.current_timestamp())
        .withColumn(model_version_col, F.lit(champion_model_version))
        .withColumn(label_column, F.lit(None).cast("long"))
        .withColumn(prediction_column, F.col(prediction_column).cast("long"))
        .select(
            *base_columns,
            label_column,
            prediction_column,
            prob_churn_column,
            model_version_col,
            inference_timestamp_col
        )
    )

    (
        DeltaTable.forName(spark, inference_enriched_table)
                  .alias("target")
                  .merge(
                      scored_df.alias("source"),
                      f"target.{customer_id_column} = source.{customer_id_column}"
                  )
                  .whenNotMatchedInsertAll()
                  .execute()
    )

    n_churn_pred = scored_df.filter(F.col(prediction_column) == 1).count()
    n_active_pred = scored_df.filter(F.col(prediction_column) == 0).count()

    print(f"Inserted {n_new:,} new predictions into {inference_enriched_table}")
    print(f"Predicted churn: {n_churn_pred:,} ({100 * n_churn_pred / n_new:.2f}%)")
    print(f"Predicted active: {n_active_pred:,} ({100 * n_active_pred / n_new:.2f}%)")
else:
    print("No new transactions to score.")

No new transactions to score.



## 7. Propagación de etiquetas reales

Se leen de `silver_churn_events` únicamente las etiquetas de clientes de producción (posteriores a `production_start_date`) cuyo `customer_id` no tiene todavía una etiqueta confirmada en `gold_churn_inference_enriched`. El resultado esperado es cero etiquetas pendientes en las primeras ejecuciones, ya que los churns se confirman con retraso una vez que el equipo de revisión cierra cada caso. El `MERGE` actualiza `label_will_churn` en las filas que ya existen en la tabla. Las filas sin etiqueta confirmada permanecen con `label_will_churn` nulo hasta que se confirmen en una ejecución posterior.

In [0]:
already_labelled_df = (
    DeltaTable.forName(spark, inference_enriched_table)
              .toDF()
              .filter(F.col(label_column).isNotNull())
              .select(customer_id_column)
)

pending_labels_df = (
    spark.table(churn_labels_table)
         .filter(F.col(date_column) >= production_start_date)
         .join(already_labelled_df, on = customer_id_column, how = "left_anti")
         .select(customer_id_column, label_column)
)

n_pending_total = pending_labels_df.count()
n_pending_churn = pending_labels_df.filter(F.col(label_column) == 1).count()
n_pending_active = pending_labels_df.filter(F.col(label_column) == 0).count()

print(f"Labels pending propagation: {n_pending_total:,}")
print(f"Churn: {n_pending_churn:,}")
print(f"Active: {n_pending_active:,}")

Labels pending propagation: 0
Churn: 0
Active: 0


In [0]:
(
    DeltaTable.forName(spark, inference_enriched_table)
              .alias("target")
              .merge(
                  pending_labels_df.alias("source"),
                  f"target.{customer_id_column} = source.{customer_id_column}"
              )
              .whenMatchedUpdate(
                  set = {f"target.{label_column}": f"source.{label_column}"}
              )
              .execute()
)

print(f"MERGE completed. {n_pending_total:,} labels propagated to {inference_enriched_table}")

MERGE completed. 0 labels propagated to workspace.credit_card_churn.gold_churn_inference_enriched


## 8. Conclusiones y siguientes pasos

### ¿Qué hace esta libreta?

1. **Carga del modelo `champion`**: Recupera el pipeline `Spark MLlib` directamente desde el alias `champion` de `Unity Catalog` y la versión concreta para trazabilidad.
2. **Inicialización de la tabla**: Crea `gold_churn_inference_enriched` vacía con el esquema correcto si es la primera ejecución, copiando la estructura de `gold_churn_test_baseline` que garantiza compatibilidad con el monitor, y activa `delta.enableChangeDataFeed` para `Databricks Lakehouse Monitoring`.
3. **Lectura de clientes pendientes**: Filtra `gold_churn_spine` para considerar únicamente los clientes posteriores a `production_start_date` e identifica mediante un `LEFT ANTI JOIN` los que todavía no tienen predicción en `gold_churn_inference_enriched`. La operación es idempotente.
4. **Enriquecimiento con el *feature store***: Replica el `PiT` *join* de entrenamiento contra `gold_customer_profile` y `gold_customer_aggregations` con las mismas `feature_names` definidas en `09_Utils.py`, garantizando que no hay *training-serving skew*.
5. **Inferencia en lote distribuida**: Aplica `.transform()` del pipeline `Spark MLlib` sobre el lote de clientes nuevos e inserta el resultado en `gold_churn_inference_enriched` con `label_will_churn` nulo.
6. **Propagación de etiquetas**: Actualiza `label_will_churn` en `gold_churn_inference_enriched` para los clientes cuya etiqueta real ha llegado a `silver_churn_events`, filtrando también por `production_start_date`.

### ¿Por qué es necesaria?

`Databricks Lakehouse Monitoring` necesita las etiquetas reales para calcular métricas de rendimiento en producción (*AUC-PR*, *F1-score*, precisión, exhaustividad) y métricas de equidad sobre los grupos de interés. Sin este paso, el monitor solo puede calcular *data drift* de características pero no detectar degradación real del modelo ni sesgos en subgrupos.

### ¿Cuándo se ejecuta?

#### Primera ejecución (manual)

1. Ejecutar la libreta manualmente para que se cree e inicialice `gold_churn_inference_enriched` con el esquema correcto.
2. Navegar a la tabla en `Catalog Explorer`, abrir la pestaña `Quality` y hacer clic en `Create monitor`.
3. Configurar el monitor con los siguientes parámetros: tipo de problema `Classification`, columna temporal `inference_timestamp`, identificador de modelo `model_version`, columna de predicción `prediction`, columna de etiqueta `label_will_churn` y tabla de referencia `gold_churn_test_baseline`.
4. Configurar las expresiones de *slice* como se detalla a continuación y hacer clic en `Create`.

#### Ejecuciones posteriores (automáticas)

Como tarea `Run_Churn_Inference_And_Label_Enrichment` en el trabajo `Churn Feature Pipeline`, que corre periódicamente después de `Publish_to_Online_Store`, para garantizar que las etiquetas generadas en el ciclo actual ya están disponibles en `silver_churn_events`.

#### Expresiones de *slice* para el monitor

`Databricks Lakehouse Monitoring` calcula todas las métricas de rendimiento y *drift* tanto sobre el total de clientes como, de forma independiente, sobre cada *slice* declarado. Esto permite detectar degradaciones que solo afectan a un subgrupo concreto aunque las métricas globales sigan siendo buenas.

Hay dos tipos de *slices* relevantes para este problema:

##### *Slices* operativos

Subgrupos definidos por características del cliente que concentran la mayor parte del churn. El monitor calcula métricas de rendimiento separadas para cada uno, lo que permite detectar si el modelo se degrada antes en los clientes de mayor riesgo:

| Expresión | Justificación |
|---|---|
| `contract_risk_group = 'high'` | Los clientes de alto riesgo contractual tienen una tasa de churn estructuralmente más alta. Una degradación en este subgrupo es especialmente crítica. |
| `contract_type = 'month-to-month'` | Los clientes sin contrato de larga duración tienen mayor propensión a abandonar. Cualquier caída de rendimiento aquí es una señal de alerta crítica. |
| `days_payment_late > 0` | Clientes con retrasos en el pago son un indicador fuerte de riesgo de churn inminente. |

##### *Slices* de equidad

Subgrupos definidos por atributos del cliente. `Databricks Lakehouse Monitoring` calcula automáticamente métricas de equidad para cada uno cuando el tipo de problema es `Classification` y hay columna de etiqueta. El objetivo es detectar si el modelo genera tasas de falsos positivos desproporcionadas en algún grupo:

| Expresión | Justificación |
|---|---|
| `gender` | Detecta si la tasa de falsos positivos difiere sistemáticamente entre grupos de género. |
| `age_group` | Detecta si el modelo es más agresivo etiquetando como churn a clientes jóvenes o mayores. |
| `contract_risk_group` | Detecta si los clientes de alto riesgo contractual reciben más falsos positivos que los de bajo riesgo. |
| `region_type` | Detecta si hay disparidad geográfica en las métricas entre clientes urbanos y rurales. |

Las métricas de equidad generadas automáticamente por el monitor para cada uno de estos *slices* son la **igualdad de oportunidades** (diferencia en tasa de verdaderos positivos entre grupos), la **paridad predictiva** (diferencia en precisión entre grupos) y la **paridad estadística** (diferencia en tasa de predicciones positivas entre grupos). Estas métricas están disponibles en la tabla `gold_churn_inference_enriched_drift_metrics` generada por el monitor.